# LMP Data Validation — PJM Western Hub

**Source:** `{schema}.staging_v1_pjm_lmps_hourly` via `lmps_hourly.pull(market=...)`  
**Hub:** WESTERN HUB  
**Markets:** DA (Day-Ahead) and RT (Real-Time)  
**Purpose:** Validate raw hourly LMP data before it enters the like-day feature pipeline.

### Checks
1. **Completeness** — Date range, hourly coverage, missing days, duplicates
2. **Nulls & Outliers** — Column-level null counts, z-score outliers, physical bounds
3. **Distributions** — Histograms for LMP components, monthly boxplots
4. **LMP Component Decomposition** — Verify `lmp_total = energy + congestion + loss`
5. **Temporal Continuity** — Day-over-day jumps, gap detection
6. **DA vs RT Comparison** — DART spread analysis
7. **Diurnal Profile** — Seasonal hourly shapes, on-peak vs off-peak spread
8. **Feature Sanity Check** — Run `lmp_features.build()`, correlation heatmap
9. **Recent Spot Check** — Last 7 days summary, last 3 days hourly profiles
10. **Validation Summary** — Pass/fail checks

## 1. Setup & Data Pull

In [1]:
import sys
from pathlib import Path
_BACKEND = str(Path().resolve().parent.parent.parent.parent)
if _BACKEND not in sys.path:
    sys.path.insert(0, _BACKEND)

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import src.like_day_forecast.settings
from src.like_day_forecast.data import lmps_hourly
from src.like_day_forecast import configs

HOURS = list(range(1, 25))
ONPEAK_HOURS = list(range(8, 24))
OFFPEAK_HOURS = list(range(1, 8)) + [24]

In [2]:
# Pull DA and RT LMP data
df_da = lmps_hourly.pull(market="da")
df_da["date"] = pd.to_datetime(df_da["date"])

df_rt = lmps_hourly.pull(market="rt")
df_rt["date"] = pd.to_datetime(df_rt["date"])

LMP_COLS = ["lmp_total", "lmp_system_energy_price", "lmp_congestion_price", "lmp_marginal_loss_price"]

for label, df in [("DA", df_da), ("RT", df_rt)]:
    print(f"--- {label} ---")
    print(f"  Shape: {df.shape}")
    print(f"  Date range: {df['date'].min().date()} to {df['date'].max().date()}")
    print(f"  Unique dates: {df['date'].nunique():,}")
    print(f"  Columns: {list(df.columns)}")
    print()

2026-02-25 18:58:14 | ℹ️  INFO | lmps_hourly.py:pull:23 | Pulling LMP hourly: WESTERN HUB (da) from dbt_pjm_v1_2026_feb_19
2026-02-25 18:58:18 | ℹ️  INFO | lmps_hourly.py:pull:27 | Pulled 106,608 rows
2026-02-25 18:58:18 | ℹ️  INFO | lmps_hourly.py:pull:23 | Pulling LMP hourly: WESTERN HUB (rt) from dbt_pjm_v1_2026_feb_19
2026-02-25 18:58:25 | ℹ️  INFO | lmps_hourly.py:pull:27 | Pulled 106,519 rows
--- DA ---
  Shape: (106608, 8)
  Date range: 2014-01-01 to 2026-02-26
  Unique dates: 4,440
  Columns: ['date', 'hour_ending', 'hub', 'market', 'lmp_total', 'lmp_system_energy_price', 'lmp_congestion_price', 'lmp_marginal_loss_price']

--- RT ---
  Shape: (106519, 8)
  Date range: 2014-01-01 to 2026-02-25
  Unique dates: 4,439
  Columns: ['date', 'hour_ending', 'hub', 'market', 'lmp_total', 'lmp_system_energy_price', 'lmp_congestion_price', 'lmp_marginal_loss_price']



### Previous 3 Days — DA, RT, and DART

Quick visual inspection of the most recent 3 days across all LMP components: Day-Ahead, Real-Time, and the DART spread.

In [ ]:
# Previous 3 days — DA hourly LMP
last_3_da = sorted(df_da["date"].unique())[-3:]
df_da_last3 = df_da[df_da["date"].isin(last_3_da)].sort_values(["date", "hour_ending"])
print(f"=== DA LMP — Previous 3 Days ===")
print(f"Dates: {[str(d.date()) for d in last_3_da]}")
display(df_da_last3[["date", "hour_ending", "lmp_total", "lmp_system_energy_price",
                      "lmp_congestion_price", "lmp_marginal_loss_price"]])

# Previous 3 days — RT hourly LMP
last_3_rt = sorted(df_rt["date"].unique())[-3:]
df_rt_last3 = df_rt[df_rt["date"].isin(last_3_rt)].sort_values(["date", "hour_ending"])
print(f"\n=== RT LMP — Previous 3 Days ===")
print(f"Dates: {[str(d.date()) for d in last_3_rt]}")
display(df_rt_last3[["date", "hour_ending", "lmp_total", "lmp_system_energy_price",
                      "lmp_congestion_price", "lmp_marginal_loss_price"]])

# Previous 3 days — DART spread (DA - RT daily flat)
overlap_dates = sorted(set(last_3_da) & set(last_3_rt))
if overlap_dates:
    da_flat = df_da[df_da["date"].isin(overlap_dates)].groupby("date")["lmp_total"].mean().rename("da_flat")
    rt_flat = df_rt[df_rt["date"].isin(overlap_dates)].groupby("date")["lmp_total"].mean().rename("rt_flat")
    dart_last3 = pd.concat([da_flat, rt_flat], axis=1).dropna()
    dart_last3["dart_spread"] = dart_last3["da_flat"] - dart_last3["rt_flat"]
    print(f"\n=== DART Spread (DA - RT Daily Flat) — Previous 3 Days ===")
    display(dart_last3.round(2))
else:
    print("\nNo overlapping dates for DART spread")

## 2. Completeness Checks

Verify every date has 24 hourly observations, identify date gaps, and check for duplicates. Both DA and RT.

In [ ]:
for label, df in [("DA", df_da), ("RT", df_rt)]:
    print(f"{'='*60}")
    print(f"  {label} Completeness")
    print(f"{'='*60}")

    # 2a. Hours per day — flag any day with != 24 hours
    hours_per_day = df.groupby("date")["hour_ending"].nunique().rename("n_hours")
    incomplete = hours_per_day[hours_per_day != 24]

    print(f"Total dates: {len(hours_per_day):,}")
    print(f"Complete (24h): {(hours_per_day == 24).sum():,}")
    print(f"Incomplete: {len(incomplete)}")

    if len(incomplete) > 0:
        print(f"\nIncomplete days (showing up to 20):")
        display(incomplete.tail(20).to_frame())

    # 2b. Date sequence gaps
    all_dates = pd.date_range(df["date"].min(), df["date"].max(), freq="D")
    present_dates = set(df["date"].dt.normalize().unique())
    missing_dates = sorted(set(all_dates) - present_dates)

    print(f"\nExpected dates in range: {len(all_dates):,}")
    print(f"Present dates: {len(present_dates):,}")
    print(f"Missing dates: {len(missing_dates)}")

    if 0 < len(missing_dates) <= 30:
        print("Missing:")
        for d in missing_dates:
            print(f"  {d.date()}")
    elif len(missing_dates) > 30:
        print(f"First 10 missing: {[d.date() for d in missing_dates[:10]]}")
        print(f"Last 10 missing:  {[d.date() for d in missing_dates[-10:]]}")

    # 2c. Duplicates
    dupes = df.duplicated(subset=["date", "hour_ending"], keep=False)
    print(f"\nDuplicate (date, hour_ending) rows: {dupes.sum()}")
    print()

# Store for validation summary
da_hours_per_day = df_da.groupby("date")["hour_ending"].nunique().rename("n_hours")
da_all_dates = pd.date_range(df_da["date"].min(), df_da["date"].max(), freq="D")
da_present_dates = set(df_da["date"].dt.normalize().unique())
da_missing_dates = sorted(set(da_all_dates) - da_present_dates)
da_dupes = df_da.duplicated(subset=["date", "hour_ending"], keep=False).sum()

## 3. Null & Outlier Analysis

Null counts per LMP component, descriptive statistics, z-score outliers (|z|>4), physical bounds [-500, 5000], and constant-value days.

In [ ]:
for label, df in [("DA", df_da), ("RT", df_rt)]:
    print(f"{'='*60}")
    print(f"  {label} Null & Outlier Analysis")
    print(f"{'='*60}")

    # Null summary
    null_summary = pd.DataFrame({
        "null_count": df[LMP_COLS].isnull().sum(),
        "null_pct": (df[LMP_COLS].isnull().sum() / len(df) * 100).round(3),
    })
    print("\nNull Summary:")
    display(null_summary)

    # Descriptive statistics
    print("\nDescriptive Statistics:")
    display(df[LMP_COLS].describe().round(2))

    # Z-score outlier detection (|z| > 4)
    print("\nOutlier Check (|z| > 4):")
    for col in LMP_COLS:
        series = df[col].dropna()
        z = (series - series.mean()) / series.std()
        n_outliers = (z.abs() > 4).sum()
        print(f"  {col}: {n_outliers} outliers  (range [{series.min():.2f}, {series.max():.2f}])")

    # Physical bounds check [-500, 5000]
    print("\n--- Physical Bounds [-500, 5000] ---")
    below = (df["lmp_total"] < -500).sum()
    above = (df["lmp_total"] > 5000).sum()
    print(f"  lmp_total < -$500:  {below}")
    print(f"  lmp_total > $5,000: {above}")

    # Constant-value days (all 24 hours same lmp_total)
    daily_std = df.groupby("date")["lmp_total"].std()
    constant_days = daily_std[daily_std == 0]
    print(f"\nConstant-value days (std=0 across 24h): {len(constant_days)}")
    if len(constant_days) > 0:
        display(constant_days.head(10).to_frame())

    print()

## 4. Distributions

Histograms for all four LMP components and monthly boxplot of DA lmp_total.

In [ ]:
# Histograms for all 4 LMP components (DA)
titles = ["lmp_total", "lmp_system_energy_price", "lmp_congestion_price", "lmp_marginal_loss_price"]
fig = make_subplots(rows=2, cols=2, subplot_titles=titles)

for i, col in enumerate(titles):
    row, col_idx = divmod(i, 2)
    fig.add_trace(
        go.Histogram(x=df_da[col], nbinsx=100, name=col, showlegend=False),
        row=row + 1, col=col_idx + 1,
    )

fig.update_layout(height=600, title_text="DA LMP Component Distributions — Western Hub")
fig.show()

In [ ]:
# Monthly boxplot of DA lmp_total
df_da["month"] = df_da["date"].dt.month
month_names = {1:"Jan",2:"Feb",3:"Mar",4:"Apr",5:"May",6:"Jun",
               7:"Jul",8:"Aug",9:"Sep",10:"Oct",11:"Nov",12:"Dec"}
df_da["month_name"] = df_da["month"].map(month_names)

fig = px.box(df_da, x="month_name", y="lmp_total",
             category_orders={"month_name": list(month_names.values())},
             title="Monthly DA LMP Distribution — Western Hub (all years)",
             labels={"lmp_total": "LMP Total ($/MWh)", "month_name": "Month"})
fig.show()

## 5. LMP Component Decomposition

Verify the identity: `lmp_total = lmp_system_energy_price + lmp_congestion_price + lmp_marginal_loss_price`. Plot residual time series and stacked monthly component shares.

In [ ]:
# Verify component identity: lmp_total ≈ energy + congestion + loss
df_da["component_sum"] = (
    df_da["lmp_system_energy_price"]
    + df_da["lmp_congestion_price"]
    + df_da["lmp_marginal_loss_price"]
)
df_da["residual"] = df_da["lmp_total"] - df_da["component_sum"]

print(f"Residual (lmp_total - sum of components):")
print(f"  Mean:   {df_da['residual'].mean():.6f}")
print(f"  Std:    {df_da['residual'].std():.6f}")
print(f"  Min:    {df_da['residual'].min():.6f}")
print(f"  Max:    {df_da['residual'].max():.6f}")
print(f"  |residual| > 0.01: {(df_da['residual'].abs() > 0.01).sum()}")

# Residual time series
fig = px.scatter(df_da, x="date", y="residual",
                 title="LMP Component Decomposition Residual (lmp_total - energy - congestion - loss)",
                 labels={"residual": "Residual ($/MWh)", "date": "Date"},
                 opacity=0.3)
fig.add_hline(y=0, line_dash="dot", line_color="gray")
fig.update_traces(marker=dict(size=2))
fig.show()

In [ ]:
# Stacked area: monthly average component shares
df_da["year_month"] = df_da["date"].dt.to_period("M")

monthly_components = df_da.groupby("year_month").agg(
    energy=("lmp_system_energy_price", "mean"),
    congestion=("lmp_congestion_price", "mean"),
    loss=("lmp_marginal_loss_price", "mean"),
).reset_index()
monthly_components["year_month"] = monthly_components["year_month"].dt.to_timestamp()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=monthly_components["year_month"], y=monthly_components["energy"],
    name="System Energy", stackgroup="one", line=dict(width=0.5),
))
fig.add_trace(go.Scatter(
    x=monthly_components["year_month"], y=monthly_components["congestion"],
    name="Congestion", stackgroup="one", line=dict(width=0.5),
))
fig.add_trace(go.Scatter(
    x=monthly_components["year_month"], y=monthly_components["loss"],
    name="Marginal Loss", stackgroup="one", line=dict(width=0.5),
))
fig.update_layout(
    title="Monthly Average LMP Component Breakdown — DA Western Hub",
    xaxis_title="Month",
    yaxis_title="$/MWh",
    height=500,
)
fig.show()

## 6. Temporal Continuity

Daily average LMP time series and day-over-day change. Flag jumps > $50.

In [ ]:
# Daily average LMP time series + day-over-day change
daily_avg = df_da.groupby("date")["lmp_total"].mean().rename("lmp_daily_avg").sort_index()
daily_change = daily_avg.diff().rename("lmp_daily_change")

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=["DA Daily Average LMP — Western Hub",
                                    "Day-over-Day Change ($/MWh)"])

fig.add_trace(go.Scatter(x=daily_avg.index, y=daily_avg.values, mode="lines",
                          line=dict(width=0.5, color="steelblue"), name="Daily Avg LMP"),
              row=1, col=1)

fig.add_trace(go.Scatter(x=daily_change.index, y=daily_change.values, mode="lines",
                          line=dict(width=0.5, color="steelblue"), name="Daily Change"),
              row=2, col=1)

# Flag extreme jumps > $50
extreme_mask = daily_change.abs() > 50
if extreme_mask.sum() > 0:
    fig.add_trace(go.Scatter(x=daily_change.index[extreme_mask],
                              y=daily_change.values[extreme_mask],
                              mode="markers", marker=dict(color="red", size=6),
                              name=f"Jump >$50: {extreme_mask.sum()}"),
                  row=2, col=1)

fig.update_layout(height=600, showlegend=True)
fig.update_yaxes(title_text="LMP ($/MWh)", row=1, col=1)
fig.update_yaxes(title_text="Change ($/MWh)", row=2, col=1)
fig.show()

print(f"Day-over-day change: mean=${daily_change.mean():.2f}, std=${daily_change.std():.2f}")
print(f"  Min: ${daily_change.min():.1f} on {daily_change.idxmin().date()}")
print(f"  Max: ${daily_change.max():.1f} on {daily_change.idxmax().date()}")
print(f"  |change| > $50: {(daily_change.abs() > 50).sum()} days")
print(f"  |change| > $100: {(daily_change.abs() > 100).sum()} days")

## 7. DA vs RT Comparison

DART spread (DA - RT daily flat), scatter plot DA vs RT, and monthly DART spread boxplot.

In [ ]:
# Daily flat averages for DA and RT
da_daily = df_da.groupby("date")["lmp_total"].mean().rename("da_flat")
rt_daily = df_rt.groupby("date")["lmp_total"].mean().rename("rt_flat")

dart = pd.concat([da_daily, rt_daily], axis=1).dropna()
dart["dart_spread"] = dart["da_flat"] - dart["rt_flat"]

print(f"DART Spread (DA - RT daily flat):")
print(f"  Observations: {len(dart):,}")
print(f"  Mean:  ${dart['dart_spread'].mean():.2f}")
print(f"  Std:   ${dart['dart_spread'].std():.2f}")
print(f"  Min:   ${dart['dart_spread'].min():.2f}")
print(f"  Max:   ${dart['dart_spread'].max():.2f}")

# DA-RT correlation
da_rt_corr = dart[["da_flat", "rt_flat"]].corr().iloc[0, 1]
print(f"\n  DA vs RT correlation: r = {da_rt_corr:.4f}")

# DART spread time series
fig = px.scatter(dart.reset_index(), x="date", y="dart_spread",
                 title="DART Spread (DA - RT) Daily Flat — Western Hub",
                 labels={"dart_spread": "DART Spread ($/MWh)", "date": "Date"},
                 opacity=0.4)
fig.add_hline(y=0, line_dash="dot", line_color="gray")
fig.update_traces(marker=dict(size=2))
fig.show()

In [ ]:
# Scatter: DA vs RT daily flat
fig = px.scatter(dart.reset_index(), x="rt_flat", y="da_flat",
                 title=f"DA vs RT Daily Flat LMP (r = {da_rt_corr:.3f}) — Western Hub",
                 labels={"da_flat": "DA LMP ($/MWh)", "rt_flat": "RT LMP ($/MWh)"},
                 opacity=0.3)
# 45-degree line
axis_min = min(dart["da_flat"].min(), dart["rt_flat"].min())
axis_max = max(dart["da_flat"].max(), dart["rt_flat"].max())
fig.add_trace(go.Scatter(x=[axis_min, axis_max], y=[axis_min, axis_max],
                          mode="lines", line=dict(dash="dot", color="gray"),
                          name="1:1 line", showlegend=True))
fig.update_traces(marker=dict(size=3), selector=dict(mode="markers"))
fig.show()

In [ ]:
# Monthly DART spread boxplot
dart_reset = dart.reset_index()
dart_reset["month"] = dart_reset["date"].dt.month
dart_reset["month_name"] = dart_reset["month"].map(month_names)

fig = px.box(dart_reset, x="month_name", y="dart_spread",
             category_orders={"month_name": list(month_names.values())},
             title="Monthly DART Spread Distribution — Western Hub",
             labels={"dart_spread": "DART Spread ($/MWh)", "month_name": "Month"})
fig.add_hline(y=0, line_dash="dot", line_color="gray")
fig.show()

## 8. Diurnal Profile

Average hourly LMP shape by season and on-peak vs off-peak spread over time.

In [ ]:
# Average hourly LMP shape by season
df_da["season"] = df_da["month"].map(
    {12:"Winter",1:"Winter",2:"Winter",3:"Spring",4:"Spring",5:"Spring",
     6:"Summer",7:"Summer",8:"Summer",9:"Fall",10:"Fall",11:"Fall"})

diurnal = df_da.groupby(["season", "hour_ending"])["lmp_total"].mean().reset_index()

fig = px.line(diurnal, x="hour_ending", y="lmp_total", color="season",
              category_orders={"season": ["Winter", "Spring", "Summer", "Fall"]},
              markers=True,
              title="Average Diurnal LMP Profile by Season — DA Western Hub",
              labels={"lmp_total": "Mean LMP ($/MWh)", "hour_ending": "Hour Ending"})
fig.update_layout(xaxis=dict(dtick=1))
fig.show()

In [ ]:
# On-peak vs off-peak spread over time
onpeak_daily = df_da[df_da["hour_ending"].isin(ONPEAK_HOURS)].groupby("date")["lmp_total"].mean().rename("onpeak")
offpeak_daily = df_da[df_da["hour_ending"].isin(OFFPEAK_HOURS)].groupby("date")["lmp_total"].mean().rename("offpeak")

peak_spread = pd.concat([onpeak_daily, offpeak_daily], axis=1).dropna()
peak_spread["spread"] = peak_spread["onpeak"] - peak_spread["offpeak"]

# Rolling 30-day average for readability
peak_spread["spread_30d"] = peak_spread["spread"].rolling(30, min_periods=1).mean()

fig = go.Figure()
fig.add_trace(go.Scatter(x=peak_spread.index, y=peak_spread["spread"],
                          mode="lines", line=dict(width=0.3, color="lightblue"),
                          name="Daily Spread", opacity=0.5))
fig.add_trace(go.Scatter(x=peak_spread.index, y=peak_spread["spread_30d"],
                          mode="lines", line=dict(width=2, color="steelblue"),
                          name="30-Day Rolling Avg"))
fig.add_hline(y=0, line_dash="dot", line_color="gray")
fig.update_layout(
    title="On-Peak vs Off-Peak LMP Spread — DA Western Hub",
    xaxis_title="Date",
    yaxis_title="On-Peak minus Off-Peak ($/MWh)",
    height=450,
)
fig.show()

print(f"Peak spread: mean=${peak_spread['spread'].mean():.2f}, std=${peak_spread['spread'].std():.2f}")
print(f"  Min: ${peak_spread['spread'].min():.1f}")
print(f"  Max: ${peak_spread['spread'].max():.1f}")

## 9. Feature Sanity Check

Run `lmp_features.build(df_da, df_rt)` and inspect shape, columns, correlation heatmap. Flag highly correlated pairs (|r| > 0.95).

In [ ]:
from src.like_day_forecast.features import lmp_features

df_feat = lmp_features.build(df_da, df_rt)
df_feat["date"] = pd.to_datetime(df_feat["date"])

print(f"LMP features shape: {df_feat.shape}")
print(f"Columns ({len(df_feat.columns) - 1} features + date):")
for col in df_feat.columns:
    print(f"  {col}")
print()
display(df_feat.describe().round(3))

In [ ]:
# Correlation heatmap of LMP features
feat_cols = [c for c in df_feat.columns if c != "date"]
corr = df_feat[feat_cols].corr()

fig = px.imshow(corr, text_auto=".2f", color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
                title="LMP Feature Correlation Matrix", aspect="auto")
fig.update_layout(height=900, width=1000)
fig.show()

# Flag highly correlated pairs (|r| > 0.95)
print("Highly correlated feature pairs (|r| > 0.95):")
high_corr_pairs = []
for i in range(len(feat_cols)):
    for j in range(i + 1, len(feat_cols)):
        r = corr.iloc[i, j]
        if abs(r) > 0.95:
            high_corr_pairs.append((feat_cols[i], feat_cols[j], r))
            print(f"  {feat_cols[i]} <-> {feat_cols[j]}: r = {r:.3f}")

if not high_corr_pairs:
    print("  None found.")

## 10. Recent Spot Check

Last 7 days summary table and last 3 days hourly LMP profiles.

In [ ]:
# Last 7 days summary
daily_summary = df_da.groupby("date").agg(
    flat=("lmp_total", "mean"),
    min=("lmp_total", "min"),
    max=("lmp_total", "max"),
    std=("lmp_total", "std"),
    n_hours=("hour_ending", "nunique"),
).sort_index()

# Add on-peak/off-peak
onpeak_summary = df_da[df_da["hour_ending"].isin(ONPEAK_HOURS)].groupby("date")["lmp_total"].mean().rename("onpeak")
offpeak_summary = df_da[df_da["hour_ending"].isin(OFFPEAK_HOURS)].groupby("date")["lmp_total"].mean().rename("offpeak")
daily_summary = daily_summary.join(onpeak_summary).join(offpeak_summary)

recent_7 = daily_summary.tail(7).copy()
recent_7.index = recent_7.index.map(lambda d: d.strftime("%Y-%m-%d (%a)"))

print("Recent 7-Day DA LMP Summary — Western Hub")
display(recent_7.round(2))

In [ ]:
# Last 3 days hourly profiles
last_3_dates = sorted(df_da["date"].unique())[-3:]
df_recent = df_da[df_da["date"].isin(last_3_dates)].copy()
df_recent["date_str"] = df_recent["date"].dt.strftime("%Y-%m-%d (%a)")

fig = px.line(df_recent, x="hour_ending", y="lmp_total", color="date_str", markers=True,
              title="Recent Hourly DA LMP Profiles — Western Hub",
              labels={"lmp_total": "LMP ($/MWh)", "hour_ending": "Hour Ending", "date_str": "Date"})
fig.update_layout(xaxis=dict(dtick=1))
fig.show()

## 11. Validation Summary

Automated pass/fail checks for data quality.

In [ ]:
# Compute all validation metrics
n_da_dates = df_da["date"].nunique()
n_da_complete = (da_hours_per_day == 24).sum()
n_da_nulls_total = df_da["lmp_total"].isnull().sum()
n_da_missing_dates = len(da_missing_dates)
n_da_dupes = da_dupes
max_residual = df_da["residual"].abs().max()
da_constant_days = df_da.groupby("date")["lmp_total"].std()
n_constant = (da_constant_days == 0).sum()
da_lmp_min = df_da["lmp_total"].min()
da_lmp_max = df_da["lmp_total"].max()

checks = [
    ("Date range covers 2014 (EXTENDED_FEATURE_START)",
     df_da["date"].min().date() <= pd.Timestamp("2014-01-01").date(),
     f"{df_da['date'].min().date()} to {df_da['date'].max().date()}"),

    ("All dates have 24 hours",
     n_da_complete >= n_da_dates - 1,
     f"{n_da_complete}/{n_da_dates} complete"),

    ("No missing calendar dates",
     n_da_missing_dates == 0,
     f"{n_da_missing_dates} missing"),

    ("No duplicate (date, hour_ending) rows",
     n_da_dupes == 0,
     f"{n_da_dupes} duplicates"),

    ("Zero nulls in lmp_total",
     n_da_nulls_total == 0,
     f"{n_da_nulls_total} nulls"),

    ("Component identity holds (|residual| < 0.01)",
     max_residual < 0.01,
     f"Max |residual| = {max_residual:.6f}"),

    ("No constant-value days (all 24h same lmp_total)",
     n_constant == 0,
     f"{n_constant} constant days"),

    ("DA-RT correlation > 0.5",
     da_rt_corr > 0.5,
     f"r = {da_rt_corr:.4f}"),

    ("LMP in physical bounds [-500, 5000]",
     da_lmp_min >= -500 and da_lmp_max <= 5000,
     f"Range: [{da_lmp_min:.2f}, {da_lmp_max:.2f}]"),
]

print("=" * 80)
print("  LMP DATA VALIDATION SUMMARY — DA Western Hub")
print("=" * 80)
all_pass = True
for name, passed, detail in checks:
    status = "PASS" if passed else "FAIL"
    if not passed:
        all_pass = False
    print(f"  [{status}] {name}")
    print(f"         {detail}")
print("=" * 80)
print(f"  {'ALL CHECKS PASSED' if all_pass else 'SOME CHECKS FAILED'}")
print("=" * 80)